## **Generación Etiquetas: Answerability**

In [ ]:
!pip install -q cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.0/319.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 54.5 MB/s eta 0:00:00


In [ ]:
def answerability(new_question: str, context: str, conversation_history: str = '') -> str:
    """Función que genera una etiqueta (ANSWERABLE, UNANSWERABLE, PARTIAL) para describir a la nueva pregunta con respecto a la conversación previa en caso de que haya."""

    base_prompt = """You are a precision-focused evaluation model. Your task is to analyze three inputs:
    1. **Context**: Provided information that may contain relevant facts or data.
    2. **Conversation History**: Previous questions and answers in the dialogue. Sometimes there is no previs dialogue.
    3. **New Question**: The latest query from the user.

    Your job is to determine **if the new question can be answered** using *only* the given context and conversation history.
    You should first defend each posture and make the case on why the question can totally, can partially and cannot be answered.
    After analyzing each posture, provide your final answer. Remember that the only information you should take into condideration is the provided one.

    **Output Format:**
    Provide your response in the following two-part format. Do not include any other text before, after, or between these parts.

    **Label:** [ANSWERABLE/UNANSWERABLE/PARTIAL/CONVERSATIONAL]
    **Explanation:** [A single, concise sentence explaining the reason for the label, based solely on the provided information.]
    **Other Postures:** ["Label", "Explanation"]

    **Label Definitions:**
    - **"ANSWERABLE"**: A question is ANSWERABLE when the provided context contains sufficient, explicit, and specific information to formulate a direct and complete response that addresses all parts of the query without requiring external knowledge or inference beyond simple clarification of pronouns or coreferences resolvable within the text.
    - **"UNANSWERABLE"**: A question is UNANSWERABLE when the provided context lacks the specific, objective information required to formulate any correct response, either because the question asks about details not mentioned, is too vague or ambiguous to interpret within the context, or seeks subjective judgment or external knowledge.
    - **"PARTIAL"**: A question is PARTIALLY answerable when the provided context contains some relevant information that addresses the general topic or a subset of the query, but lacks the specific details, evidence, or completeness required to provide a full and definitive answer.
    - **"CONVERSATIONAL"**: A CONVERSATIONAL label is applied when the user's input is not a substantive question seeking factual information from the context, but rather a social remark, procedural utterance, or conversational turn meant to manage the interaction itself.

    **Considerations:**
    - Unlike CONVERSATIONAL, an UNANSWERABLE input is a genuine (if flawed) question seeking factual information. Unlike PARTIAL, the context provides *no* relevant information that directly addresses the core query.
    - Unlike ANSWERABLE, a PARTIAL context cannot yield a complete answer. Unlike UNANSWERABLE, a PARTIAL context does provide *some* directly relevant factual information that moves toward an answer.
    - Unlike UNANSWERABLE or PARTIAL, a CONVERSATIONAL input is not attempting to query the contextual information at all. Its purpose is fundamentally different.


    **Critical Rules:**
    - **Do not answer the question itself.** Your explanation must only justify the label, not provide the answer.
    - **Do not generate any part of an answer.**
    - **Base your decision and explanation strictly on the provided inputs.**
    - **Keep the explanation brief and factual.**

**Inputs:**
"""

    if(conversation_history != ''):
        prompt = base_prompt + f"## Context: \n{context}\n## Conversation History: \n{conversation_history}\n## New Question: \n{new_question}\n\n**Output:** [Yes/Partial/No]"
    else:
        prompt = base_prompt + f"## Context: \n{context}\n## New Question: \n{new_question}\n\n**Output:** [Yes/Partial/No]"

    # Aquí vaunafuncion de inferencia, yo use una personalizadapara DeepSeek
    # el resultado se debeasignar a result
    # result = inferencia(prompt)
    result = "" # borrar esta línea una vez hecho el cambio

    if 'UNANSWERABLE' in text.split("\n")[0]:
        return 'UNANSWERABLE'
    elif 'ANSWERABLE' in text.split("\n")[0]:
        return 'ANSWERABLE'
    elif 'PARTIAL' in text.split("\n")[0]:
        return 'PARTIAL'
    else:
        return 'CONVERSATIONAL'

In [ ]:
import json
import cohere

# ============================================================
# CONFIGURACIÓN COHERE (API KEY DIRECTA)
# ============================================================
COHERE_API_KEY = "COHERE API"
co = cohere.Client(COHERE_API_KEY)
MODEL_NAME = "command-a-03-2025"

# ============================================================
# PROMPT ORIGINAL (SIN CAMBIOS)
# ============================================================
BASE_PROMPT = """You are a precision-focused evaluation model. Your task is to analyze three inputs:
1. **Context**: Provided information that may contain relevant facts or data.
2. **Conversation History**: Previous questions and answers in the dialogue. Sometimes there is no previs dialogue.
3. **New Question**: The latest query from the user.

Your job is to determine **if the new question can be answered** using *only* the given context and conversation history.
You should first defend each posture and make the case on why the question can totally, can partially and cannot be answered.
After analyzing each posture, provide your final answer. Remember that the only information you should take into condideration is the provided one.

**Output Format:**
Provide your response in the following two-part format. Do not include any other text before, after, or between these parts.

**Label:** [ANSWERABLE/UNANSWERABLE/PARTIAL/CONVERSATIONAL]
**Explanation:** [A single, concise sentence explaining the reason for the label, based solely on the provided information.]
**Other Postures:** ["Label", "Explanation"]

**Label Definitions:**
- **"ANSWERABLE"**: A question is ANSWERABLE when the provided context contains sufficient, explicit, and specific information to formulate a direct and complete response that addresses all parts of the query without requiring external knowledge or inference beyond simple clarification of pronouns or coreferences resolvable within the text.
- **"UNANSWERABLE"**: A question is UNANSWERABLE when the provided context lacks the specific, objective information required to formulate any correct response, either because the question asks about details not mentioned, is too vague or ambiguous to interpret within the context, or seeks subjective judgment or external knowledge.
- **"PARTIAL"**: A question is PARTIALLY answerable when the provided context contains some relevant information that addresses the general topic or a subset of the query, but lacks the specific details, evidence, or completeness required to provide a full and definitive answer.
- **"CONVERSATIONAL"**: A CONVERSATIONAL label is applied when the user's input is not a substantive question seeking factual information from the context, but rather a social remark, procedural utterance, or conversational turn meant to manage the interaction itself.

**Critical Rules:**
- **Do not answer the question itself.**
- **Do not generate any part of an answer.**
- **Base your decision and explanation strictly on the provided inputs.**
- **Keep the explanation brief and factual.**

**Inputs:**
"""

# ============================================================
# FUNCIÓN ANSWERABILITY (CHAT API CORRECTA)
# ============================================================
def answerability(new_question: str, context: str, conversation_history: str = "") -> str:
    """Función que genera una etiqueta (ANSWERABLE, UNANSWERABLE, PARTIAL) para describir a la nueva pregunta con respecto a la conversación previa en caso de que haya."""

    if conversation_history.strip():
        prompt = (
            BASE_PROMPT
            + f"## Context:\n{context}\n"
            + f"## Conversation History:\n{conversation_history}\n"
            + f"## New Question:\n{new_question}\n"
        )
    else:
        prompt = (
            BASE_PROMPT
            + f"## Context:\n{context}\n"
            + f"## New Question:\n{new_question}\n"
        )

    response = co.chat(
        model=MODEL_NAME,
        message=prompt,
        temperature=0.0,
        max_tokens=300
    )

    text = response.text.strip()
    first_line = text.split("\n")[0]

    if "UNANSWERABLE" in first_line:
        return "UNANSWERABLE"
    elif "ANSWERABLE" in first_line:
        return "ANSWERABLE"
    elif "PARTIAL" in first_line:
        return "PARTIAL"
    else:
        return "CONVERSATIONAL"

# ============================================================
# HISTORIAL DE CONVERSACIÓN (JSON REAL)
# ============================================================
def build_conversation_history(inputs):
    history = []
    for turn in inputs[:-1]:
        role = turn["speaker"].upper()
        history.append(f"{role}: {turn['text']}")
    return "\n".join(history)

# ============================================================
# PIPELINE PRINCIPAL
# ============================================================
INPUT_PATH = "reference_taskB.jsonl"
OUTPUT_PATH = "reference_taskB_with_answerability.jsonl"

with open(INPUT_PATH, "r", encoding="utf-8") as fin, \
     open(OUTPUT_PATH, "w", encoding="utf-8") as fout:

    for line in fin:
        record = json.loads(line)

        context = "\n\n".join(c["text"] for c in record["contexts"])
        conversation_history = build_conversation_history(record["input"])
        new_question = record["input"][-1]["text"]

        label = answerability(
            new_question=new_question,
            context=context,
            conversation_history=conversation_history
        )

        record["answerability"] = label
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")

print("✅ JSONL etiquetado correctamente:", OUTPUT_PATH)


✅ JSONL etiquetado correctamente: reference_taskB_with_answerability.jsonl


In [ ]:
import json
from collections import Counter

PATH = "reference_taskB_with_answerability.jsonl"

counts = Counter()

with open(PATH, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        counts[record["answerability"]] += 1

for label, n in counts.items():
    print(f"{label}: {n}")

PARTIAL: 226
UNANSWERABLE: 110
ANSWERABLE: 150
CONVERSATIONAL: 21
